# 📊 Notebook 2 — Data Loading & Exploratory Data Analysis
**Pill Counter | Computer Vision Pipeline**

> **Goal:** Discover the dataset, parse YOLO annotations, compute statistics, visualise class distributions and sample images, then produce a clean train/val/test split ready for preprocessing.

---
**Key outputs produced by this notebook:**
- `data/processed/` — organised split directories
- `data/processed/dataset.yaml` — YOLO-ready dataset config


## 2.1 — Imports & Config

In [1]:
import sys, yaml, logging, json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from pathlib import Path
from typing import List, Optional, Tuple
from sklearn.model_selection import train_test_split
import cv2

logging.basicConfig(level=logging.WARNING)  # suppress info noise in notebooks
sns.set_theme(style="darkgrid")
plt.rcParams["figure.dpi"] = 120

with open("config.yaml") as f:
    CFG = yaml.safe_load(f)

DATA_DIR   = Path(CFG["paths"]["raw_data"])
OUTPUT_DIR = Path(CFG["paths"]["processed_data"])
IMG_SIZE   = CFG["dataset"]["img_size"]

print(f"Raw data   : {DATA_DIR}")
print(f"Output dir : {OUTPUT_DIR}")
print(f"Image size : {IMG_SIZE}")


ModuleNotFoundError: No module named 'yaml'

## 2.2 — Dataset Discovery

In [ ]:
FORMATS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

image_paths = sorted([
    p for fmt in FORMATS
    for p in DATA_DIR.rglob(f"*{fmt}")
])

print(f"Total images found : {len(image_paths)}")

# Format breakdown
from collections import Counter
fmt_counts = Counter(p.suffix.lower() for p in image_paths)
for fmt, cnt in fmt_counts.items():
    print(f"  {fmt:8s} : {cnt}")

# Quick sanity — show first 5
print("\nSample paths:")
for p in image_paths[:5]:
    print(f"  {p}")


## 2.3 — Annotation Parsing & Validation

In [ ]:
def parse_annotation(ann_path: Path) -> Optional[List[dict]]:
    """Parse a YOLO .txt annotation into a list of box dicts."""
    if not ann_path.exists():
        return None
    boxes = []
    with open(ann_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) != 5:
                continue
            cls, xc, yc, w, h = int(parts[0]), *map(float, parts[1:])
            boxes.append({"class_id": cls, "x_center": xc, "y_center": yc, "width": w, "height": h})
    return boxes


valid_images, valid_anns, missing = [], [], []

for img_path in image_paths:
    ann_path = img_path.with_suffix(".txt")
    ann = parse_annotation(ann_path)
    if ann is not None:
        valid_images.append(img_path)
        valid_anns.append(ann)
    else:
        missing.append(img_path)

print(f"Images with annotations : {len(valid_images)}")
print(f"Images missing labels   : {len(missing)}")
print(f"Annotation coverage     : {len(valid_images)/len(image_paths)*100:.1f}%")

# Total boxes
total_boxes = sum(len(a) for a in valid_anns)
print(f"\nTotal bounding boxes    : {total_boxes}")
print(f"Avg boxes per image     : {total_boxes/max(len(valid_images),1):.2f}")


## 2.4 — Bounding Box Statistics

In [ ]:
widths, heights, areas = [], [], []
for ann in valid_anns:
    for box in ann:
        widths.append(box["width"])
        heights.append(box["height"])
        areas.append(box["width"] * box["height"])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Bounding Box Distribution (normalised coords)", fontsize=13, fontweight="bold")

axes[0].hist(widths,  bins=40, color="#4C8BF5", edgecolor="white")
axes[0].set_title("Box Width")
axes[0].set_xlabel("Normalised width")

axes[1].hist(heights, bins=40, color="#F5844C", edgecolor="white")
axes[1].set_title("Box Height")
axes[1].set_xlabel("Normalised height")

axes[2].hist(areas,   bins=40, color="#4CF584", edgecolor="white")
axes[2].set_title("Box Area (w×h)")
axes[2].set_xlabel("Normalised area")

plt.tight_layout()
plt.savefig("results/visualizations/bbox_distributions.png", bbox_inches="tight")
plt.show()

print(f"Width   — mean: {np.mean(widths):.3f}  std: {np.std(widths):.3f}  min: {min(widths):.3f}  max: {max(widths):.3f}")
print(f"Height  — mean: {np.mean(heights):.3f}  std: {np.std(heights):.3f}  min: {min(heights):.3f}  max: {max(heights):.3f}")
print(f"Area    — mean: {np.mean(areas):.4f}  std: {np.std(areas):.4f}  min: {min(areas):.4f}  max: {max(areas):.4f}")


## 2.5 — Image Resolution Analysis

In [ ]:
# Sample up to 200 images for speed
sample_paths = valid_images[:200]
resolutions = []

for p in sample_paths:
    img = cv2.imread(str(p))
    if img is not None:
        h, w = img.shape[:2]
        resolutions.append((w, h))

ws = [r[0] for r in resolutions]
hs = [r[1] for r in resolutions]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle(f"Image Resolutions (sample of {len(resolutions)})", fontsize=13, fontweight="bold")

axes[0].scatter(ws, hs, alpha=0.4, color="#4C8BF5", s=15)
axes[0].set_xlabel("Width (px)"); axes[0].set_ylabel("Height (px)")
axes[0].set_title("Width vs Height scatter")
axes[0].axline((0,0), slope=1, color="red", linestyle="--", linewidth=0.8, label="1:1 aspect")
axes[0].legend()

axes[1].hist2d(ws, hs, bins=20, cmap="Blues")
axes[1].set_xlabel("Width (px)"); axes[1].set_ylabel("Height (px)")
axes[1].set_title("Density heatmap")

plt.tight_layout()
plt.savefig("results/visualizations/resolution_analysis.png", bbox_inches="tight")
plt.show()

print(f"Width  — mean: {int(np.mean(ws))}px  range: [{min(ws)}, {max(ws)}]")
print(f"Height — mean: {int(np.mean(hs))}px  range: [{min(hs)}, {max(hs)}]")


## 2.6 — Sample Image Visualisation

In [ ]:
import random

CLASS_NAMES = ["pill"]
COLORS = [(0, 200, 80)]  # one colour per class

def draw_yolo_boxes(img_bgr, boxes, class_names):
    img = img_bgr.copy()
    h, w = img.shape[:2]
    for box in boxes:
        cls = box["class_id"]
        xc, yc, bw, bh = box["x_center"], box["y_center"], box["width"], box["height"]
        x1 = int((xc - bw/2) * w);  y1 = int((yc - bh/2) * h)
        x2 = int((xc + bw/2) * w);  y2 = int((yc + bh/2) * h)
        color = COLORS[cls % len(COLORS)]
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        label = class_names[cls] if cls < len(class_names) else str(cls)
        cv2.putText(img, label, (x1, max(y1-5, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    return img

n_show = min(8, len(valid_images))
idxs   = random.sample(range(len(valid_images)), n_show)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle("Sample Annotated Images", fontsize=13, fontweight="bold")
axes = axes.flatten()

for ax_i, img_idx in enumerate(idxs):
    img = cv2.imread(str(valid_images[img_idx]))
    if img is None:
        continue
    img_rgb = cv2.cvtColor(draw_yolo_boxes(img, valid_anns[img_idx], CLASS_NAMES), cv2.COLOR_BGR2RGB)
    ax = axes[ax_i]
    ax.imshow(img_rgb)
    ax.set_title(f"{len(valid_anns[img_idx])} pill(s)", fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.savefig("results/visualizations/sample_annotations.png", bbox_inches="tight")
plt.show()


## 2.7 — Train / Val / Test Split

In [ ]:
TRAIN_SIZE = CFG["dataset"]["train_size"]
VAL_SIZE   = CFG["dataset"]["val_size"]
SEED       = 42

train_imgs, temp_imgs, train_anns, temp_anns = train_test_split(
    valid_images, valid_anns, test_size=1-TRAIN_SIZE, random_state=SEED
)
val_ratio = VAL_SIZE / (1 - TRAIN_SIZE)
val_imgs, test_imgs, val_anns, test_anns = train_test_split(
    temp_imgs, temp_anns, test_size=1-val_ratio, random_state=SEED
)

splits = {"train": (train_imgs, train_anns),
          "val":   (val_imgs,   val_anns),
          "test":  (test_imgs,  test_anns)}

print("Split summary:")
for name, (imgs, _) in splits.items():
    print(f"  {name:6s}: {len(imgs):>5d} images")

# Pie chart
fig, ax = plt.subplots(figsize=(5, 5))
sizes = [len(v[0]) for v in splits.values()]
ax.pie(sizes, labels=list(splits.keys()), autopct="%1.1f%%",
       colors=["#4C8BF5", "#F5844C", "#4CF584"], startangle=90)
ax.set_title("Dataset Split", fontsize=12, fontweight="bold")
plt.savefig("results/visualizations/dataset_split.png", bbox_inches="tight")
plt.show()


## 2.8 — Organise Files & Write dataset.yaml

In [ ]:
import shutil

def organise_split(split_name, img_paths, ann_list, out_root):
    img_dir = out_root / split_name / "images"
    lbl_dir = out_root / split_name / "labels"
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    for img_path, ann in zip(img_paths, ann_list):
        # Copy image
        dst_img = img_dir / img_path.name
        if not dst_img.exists():
            shutil.copy2(img_path, dst_img)

        # Write annotation
        dst_lbl = lbl_dir / img_path.with_suffix(".txt").name
        with open(dst_lbl, "w") as f:
            for box in ann:
                f.write(f"{box['class_id']} {box['x_center']:.6f} {box['y_center']:.6f} "
                        f"{box['width']:.6f} {box['height']:.6f}\n")

    print(f"  ✅  {split_name}: {len(img_paths)} images written")

print("Organising dataset structure...")
for name, (imgs, anns) in splits.items():
    organise_split(name, imgs, anns, OUTPUT_DIR)

# Write dataset.yaml
dataset_yaml = {
    "path":  str(OUTPUT_DIR.absolute()),
    "train": "train/images",
    "val":   "val/images",
    "test":  "test/images",
    "nc":    1,
    "names": ["pill"],
}
yaml_path = OUTPUT_DIR / "dataset.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(dataset_yaml, f, default_flow_style=False)

print(f"\n✅  dataset.yaml written to: {yaml_path}")


## ✅  Notebook 2 Complete
> Dataset split and organised. Proceed to **Notebook 3 — Preprocessing & Augmentation**.